# 00. JupyterLab и NVIDIA GPU preflight

Выберите kernel **Python (DiatomDINO GPU)**. Этот notebook ничего не скачивает и не обучает: он проверяет, что JupyterLab использует нужное виртуальное окружение и CUDA-сборку PyTorch.

Kernel создаётся до запуска JupyterLab:
```bash
python -m ipykernel install --user --name diatom-dino --display-name "Python (DiatomDINO GPU)"
python -m jupyter lab
```

In [ ]:
from core.notebook_runtime import bootstrap_notebook, describe_runtime, gpu_preflight, run_guarded

GPU_INDEX = 0
MINIMUM_VRAM_GB = 8.0
RUN_SUBPROCESS_PREFLIGHT = False
context = bootstrap_notebook(gpu_index=GPU_INDEX)
describe_runtime(context)

## CUDA активного kernel

In [ ]:
gpu = gpu_preflight(context, minimum_vram_gb=MINIMUM_VRAM_GB)
assert gpu['cuda_available']
print('Jupyter GPU kernel: OK')

## Изолированный subprocess preflight

Обучающие notebooks запускают CLI именно этим интерпретатором. Проверка ниже подтверждает JupyterLab, CUDA, VRAM и свободное место в отдельном процессе.

In [ ]:
command = context.module_command(
    'scripts.check_environment',
    '--data-root', 'data',
    '--minimum-free-gb', '100',
    '--gpu-index', '0',  # selected physical GPU is logical cuda:0 in the child process
    '--minimum-vram-gb', str(MINIMUM_VRAM_GB),
    '--require-jupyter',
)
run_guarded(context, command, enabled=RUN_SUBPROCESS_PREFLIGHT, label='environment-preflight')

## Практические GPU-профили

При нехватке VRAM уменьшайте batch/eval batch и затем размер изображения. Состав датасета и frozen benchmark не меняются.

In [ ]:
profiles = {
    '8-11 GiB': {'yolo_batch': 2, 'yolo_imgsz': 768, 'dino_eval_batch': 16, 'workers': 2},
    '12-15 GiB': {'yolo_batch': 4, 'yolo_imgsz': 1024, 'dino_eval_batch': 32, 'workers': 4},
    '16+ GiB': {'yolo_batch': 8, 'yolo_imgsz': 1024, 'dino_eval_batch': 64, 'workers': 4},
}
profiles